In [1]:
from sand_bob._code_gen import python_code_to_beautiful_notebook

In [ ]:
res = python_code_to_beautiful_notebook("""
# Import the libraries we need (all are in the allowed whitelist)
import numpy as np
from skimage import io, filters, morphology, measure, exposure, color
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# 1. Load the image
# ----------------------------------------------------------------------
image_path = "input_data/blobs.tif"
# The image is likely grayscale; if not, we convert it to gray.
raw_image = io.imread(image_path)
if raw_image.ndim == 3:                     # RGB image -> convert to gray
    gray_image = color.rgb2gray(raw_image)
else:
    gray_image = raw_image.astype(float) / 255.0   # normalise to [0,1]

# ----------------------------------------------------------------------
# 2. Enhance contrast (optional, helps with thresholding)
# ----------------------------------------------------------------------
# Use histogram equalisation to stretch the intensity range.
enhanced = exposure.equalize_adapthist(gray_image)

# ----------------------------------------------------------------------
# 3. Find a threshold that separates bright objects from the background
# ----------------------------------------------------------------------
# Otsu's method works well for bimodal histograms.
threshold_value = filters.threshold_otsu(enhanced)
binary_mask = enhanced > threshold_value

# ----------------------------------------------------------------------
# 4. Clean the binary mask
# ----------------------------------------------------------------------
# Remove small noise and fill holes inside objects.
cleaned = morphology.remove_small_objects(binary_mask, min_size=20)
cleaned = morphology.remove_small_holes(cleaned, area_threshold=20)

# ----------------------------------------------------------------------
# 5. Label connected components (bright objects)
# ----------------------------------------------------------------------
labeled_image = measure.label(cleaned, connectivity=2)
object_count = labeled_image.max()   # number of distinct labels

# ----------------------------------------------------------------------
# 6. Visualise the results
# ----------------------------------------------------------------------
# Original image
fig1, ax1 = plt.subplots(figsize=(5,5))
ax1.imshow(gray_image, cmap='gray')
ax1.set_title("Original Gray Image")
ax1.axis('off')
plt.show()

# Binary mask after cleaning
fig2, ax2 = plt.subplots(figsize=(5,5))
ax2.imshow(cleaned, cmap='gray')
ax2.set_title("Cleaned Binary Mask")
ax2.axis('off')
plt.show()

# Labeled objects (random colormap)
fig3, ax3 = plt.subplots(figsize=(5,5))
ax3.imshow(color.label2rgb(labeled_image, bg_label=0))
ax3.set_title("Labeled Bright Objects")
ax3.axis('off')
plt.show()

# ----------------------------------------------------------------------
# 7. Report the measurement
# ----------------------------------------------------------------------
# Second‑last line: name of the measurement
display("Number of bright objects")

# Final line: the actual count (printed on a new line)
print(object_count)
""",
    dependencies=["scikit-image", "matplotlib"],
    input_host_path="input_data")

In [ ]:
list(res.files.keys())

In [ ]:
res

In [ ]:
# Write the byte array to a binary file using context manager
with open("test2.ipynb", 'wb') as file:
    file.write(res.files['/display_output/notebook_executed.ipynb'])
